# 04 · Timing, BatchNorm check, baseline  (steps 8-9, GPU session)
1. **Time first.** Everything else is planned from this number.
2. **BatchNorm check:** three short runs; micro-batch 4 must score like micro-batch 1.
3. **Baseline reproduction:** official PEdger in our pipeline (gate: within ~0.005 of the published
   number, otherwise our retrain is the declared baseline).

In [ ]:
# --- setup: code + upstream PEdger -------------------------------------------
import os, subprocess, sys
REPO_URL = ""          # e.g. "https://github.com/<you>/pedger-mtkd.git" (or attach the code as a Kaggle Dataset)
CODE_DATASET = "/kaggle/input/pedger-mtkd-code"
W = "/kaggle/working"
REPO = f"{W}/pedger-mtkd"
if not os.path.exists(REPO):
    if REPO_URL:
        subprocess.run(["git", "clone", "-q", REPO_URL, REPO], check=True)
    else:
        subprocess.run(["cp", "-r", CODE_DATASET, REPO], check=True)
os.chdir(REPO)
sys.path.insert(0, REPO)
!python tools/fetch_upstream.py
!pip install -q -r requirements.txt

In [ ]:
# --- inputs (attach these Kaggle Datasets; names are suggestions) --------------
RAW_BSDS = "/kaggle/input/bsds500-raw/BSR/BSDS500/data"     # original Berkeley release
DATA = "/kaggle/input/pedger-data"                           # output of 01 (array stores), saved as a Dataset
CACHE = "/kaggle/input/pedger-teachers"                      # output of 03 (teacher cache), saved as a Dataset
RUNS = "/kaggle/working/runs"
import os
if not os.path.exists(DATA): DATA = "/kaggle/working/data_store"
if not os.path.exists(CACHE): CACHE = "/kaggle/working/teacher_cache"

In [ ]:
!python bench/time_run.py --config configs/baseline.yaml --data-root {DATA} --teacher-root {CACHE} --out /kaggle/working/timing_baseline.json
!python bench/time_run.py --config configs/A_agree.yaml --data-root {DATA} --teacher-root {CACHE} --out /kaggle/working/timing_A_agree.json
# If peak memory allows, try a larger micro-batch (keep micro_batch x accum_steps = 16):
# !python bench/time_run.py --config configs/baseline.yaml --data-root {DATA} --teacher-root {CACHE} --set train.micro_batch=8 train.accum_steps=2

**Read the estimate.** `est_hours_full_run` x number of runs vs ~30 GPU-h/week. If one full run does
not fit the budget, shorten the schedule for **every** row in the same way (e.g. `--set data.samples_per_epoch=...`),
state it in the paper, and compare only against our own retrained baseline in that schedule.

In [ ]:
# BatchNorm check (step 9): short runs on the dev split, scored on val
for c in ["bn_micro1", "bn_micro4", "bn_stock4"]:
    !python -m pedger_kd.train_kd --config configs/checks/{c}.yaml --data-root {DATA} --teacher-root {CACHE} --out {RUNS}/{c} --max-hours 3.5

In [ ]:
for c in ["bn_micro1", "bn_micro4", "bn_stock4"]:
    !python eval/run_bsds_eval.py --run {RUNS}/{c} --store {DATA}/bsds --split val --raw-root {RAW_BSDS} --nproc 4
!python eval/bn_check.py --compare-runs {RUNS}/bn_micro1/scores_val_ns_m.json {RUNS}/bn_micro4/scores_val_ns_m.json {RUNS}/bn_stock4/scores_val_ns_m.json

In [ ]:
# Baseline seed 0 (resumable: re-run the same cell in a new session; attach the previous output as a Dataset
# and copy runs/ back into /kaggle/working first)
!python -m pedger_kd.train_kd --config configs/baseline.yaml --seed 0 --data-root {DATA} --teacher-root {CACHE} --out {RUNS}/baseline_s0 --max-hours 11.5